# Vector embedding — SigLIP2 + Qwen3-VL-Embedding 2B/8B

Notebook này tạo **3 bộ frame embeddings độc lập** để so sánh chất lượng retrieval trên cùng keyframes:

| model_key | checkpoint | dim | Kaggle execution | GCS `extractor_version` |
|---|---|---:|---|---|
| `siglip2` | `google/siglip2-so400m-patch14-384` | 1152 | 1 replica / GPU | `siglip2-so400m-patch14-384-v1` |
| `qwen3-vl-2b` | `Qwen/Qwen3-VL-Embedding-2B` | 2048 | 1 replica / GPU | `qwen3-vl-2b` |
| `qwen3-vl-8b` | `Qwen/Qwen3-VL-Embedding-8B` | 4096 | FP16 model sharded across 2 GPUs | `qwen3-vl-8b` |

Thiết kế Kaggle T4×2:
- SigLIP2 và Qwen 2B chạy **data parallel không đồng bộ gradient**: mỗi T4 xử lý một tập video riêng.
- Qwen 8B chạy **một process, shard FP16 qua cả hai T4** (`balanced_low_0`) để tránh phải quantize model weights.
- Qwen dùng `max_pixels = 256×256` mặc định để so sánh gần với input resolution của SigLIP2 và giảm chi phí visual tokens. Có thể tăng sau nếu muốn đánh giá Qwen ở native/higher resolution.
- Tất cả embedding được L2-normalize và mặc định lưu float32.
- Mỗi model upload vào một `extractor_version` riêng, nên không ghi đè nhau.

> Notebook chỉ tạo embeddings. Để kết luận model nào tốt hơn, hãy dùng cùng query set/ground truth và so Recall@K, mAP/nDCG hoặc metric của AIC; không so trực tiếp cosine score giữa ba model khác nhau.


## 1. Install dependencies

In [ ]:
%pip install -q -U \
  google-cloud-storage pandas pillow tqdm accelerate \
  "transformers>=4.57.1" "qwen-vl-utils>=0.0.14"


## 2. Configuration

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GCS_BUCKET_SECRET = user_secrets.get_secret("GCS_BUCKET")
GCS_CREDENTIALS_JSON = user_secrets.get_secret("GCS_CREDENTIALS_JSON")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Do not print secret values.
print("Secrets loaded:",
      bool(GCS_BUCKET_SECRET),
      bool(GCS_CREDENTIALS_JSON),
      bool(HF_TOKEN))


In [ ]:
from pathlib import Path
import json
import os
import torch

# -------------------------
# Local Kaggle input
# -------------------------
INPUT_ROOT = "/kaggle/input/datasets/khngxuninh/aiteam-dataset-batch-0-material-keyframes-l21-l25"

# Keep/change the batch list as needed.
# BATCHES = ["L21", "L22", "L23", "L24", "L25"]
BATCHES = ["L22"]

# None = all videos. Use 1 for a smoke test.
MAX_VIDEOS_PER_BATCH = None

# Run all three sequentially. You can temporarily select a subset while debugging.
MODELS_TO_RUN = ["siglip2", "qwen3-vl-2b", "qwen3-vl-8b"]

# -------------------------
# Fair-ish image resolution for frame retrieval comparison
# -------------------------
# SigLIP2 is fixed at 256x256. Qwen accepts dynamic resolution, so cap total
# pixels at 256*256 by default to avoid giving Qwen extra visual resolution.
QWEN_MIN_PIXELS = 4 * 32 * 32
QWEN_MAX_PIXELS = 256 * 256
QWEN_DOCUMENT_INSTRUCTION = "Represent the user's input."

MODEL_SPECS = {
    "siglip2": {
        "backend": "siglip2",
        "checkpoint": "google/siglip2-so400m-patch14-384",
        "embedding_dim": 1152,
        "batch_size_per_gpu": 512,
        "launch_mode": "replicated",
        "extractor_version": "siglip2-so400m-patch14-384-v1",
    },
    "qwen3-vl-2b": {
        "backend": "qwen3_vl",
        "checkpoint": "Qwen/Qwen3-VL-Embedding-2B",
        "embedding_dim": 2048,
        # Conservative T4 default. OOM batches are automatically split in half.
        "batch_size_per_gpu": 8,
        "launch_mode": "replicated",
        "extractor_version": "qwen3-vl-2b",
    },
    "qwen3-vl-8b": {
        "backend": "qwen3_vl",
        "checkpoint": "Qwen/Qwen3-VL-Embedding-8B",
        "embedding_dim": 4096,
        # 8B is sharded across both T4s. Start at 2; automatic OOM split -> 1.
        "batch_size_per_gpu": 2,
        "launch_mode": "sharded",
        "extractor_version": "qwen3-vl-8b",
    },
}

# -------------------------
# Data loading / persistence
# -------------------------
NUM_WORKERS_PER_GPU = None
PREFETCH_FACTOR = 2
SAVE_DTYPE = "float32"  # change to float16 only if GCS storage is more important than precision

# -------------------------
# GCS output
# -------------------------
GCS_BUCKET = GCS_BUCKET_SECRET or "aic_ai_2026"
DATASET_ID = "ai_challenge_2025"
FRAME_PROFILE = "autoshot_v1"
OUTPUT_PREFIX = "features/extractors"

UPLOAD_TO_GCS = True
SKIP_EXISTING = True
UPLOAD_WORKERS_PER_RANK = 4

GCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"
GCS_CREDENTIALS_FILE = ""

LOCAL_OUTPUT_ROOT_BASE = "/kaggle/working/vector_embeddings_3models"

GPU_COUNT = torch.cuda.device_count()
print("CUDA count:", GPU_COUNT)
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} - {torch.cuda.get_device_name(i)} - {props.total_memory/1024**3:.1f} GiB")

if GPU_COUNT < 1:
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")

if "qwen3-vl-8b" in MODELS_TO_RUN and GPU_COUNT < 2:
    raise RuntimeError(
        "qwen3-vl-8b is configured for full-FP16 sharding and needs Kaggle T4x2. "
        "Select 2 GPUs or remove qwen3-vl-8b from MODELS_TO_RUN."
    )

print("Batches:", BATCHES)
print("Models:", MODELS_TO_RUN)


## 3. Preview local folders before embedding

Cell này **chỉ nhìn local Kaggle input**, không gọi GCS và không download ảnh.


In [8]:
from pathlib import Path

root = Path(INPUT_ROOT)
print(root)

def find_batch_dirs(batch):
    candidates = []
    candidates += list(root.glob(f"*Keyframes_{batch}/keyframes"))
    candidates += list(root.glob(f"*/*Keyframes_{batch}/keyframes"))
    candidates += list(root.glob(f"*/*/*Keyframes_{batch}/keyframes"))
    return [p for p in candidates if p.is_dir()]

# Count the number of videos and frames
for batch in BATCHES:
    dirs = find_batch_dirs(batch)
    print(f"\n{batch}:")
    for d in dirs:
        videos = [p for p in d.iterdir() if p.is_dir() and p.name.startswith(batch + "_V")]
        image_count = 0
        for v in videos:
            image_count += sum(
                1 for p in v.iterdir()
                if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
            )
        print(" ", d)
        print("    videos =", len(videos), "| images =", image_count)


/kaggle/input/datasets/khngxuninh/aiteam-dataset-batch-0-material-keyframes-l21-l25

L22:
  /kaggle/input/datasets/khngxuninh/aiteam-dataset-batch-0-material-keyframes-l21-l25/002_Keyframes_L22/keyframes
    videos = 31 | images = 9096


## 4. Write the generic 3-model worker

Một worker dùng chung cho cả ba model:
- SigLIP2: `AutoImageProcessor + AutoModel.get_image_features`.
- Qwen3-VL: image-only implementation theo preprocessing/pooling của Qwen3-VL-Embedding.
- Qwen 8B hỗ trợ `device_map="balanced_low_0"` để shard FP16 qua T4×2.
- Qwen batch có **automatic OOM fallback**: nếu batch quá lớn, tự tách đôi cho tới khi chạy được.


In [ ]:
from pathlib import Path

SCRIPT_PATH = "/kaggle/working/embed_3models_kaggle.py"

script_text = 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Optional\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageFile\nfrom tqdm.auto import tqdm\n\nimport torch\nimport torch.distributed as dist\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\nVALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}\n\n\n@dataclass(frozen=True)\nclass VideoJob:\n    batch_id: str\n    video_id: str\n    video_dir: str\n    image_paths: tuple[str, ...]\n\n    @property\n    def num_frames(self) -> int:\n        return len(self.image_paths)\n\n\ndef natural_key(path: str | Path):\n    name = Path(path).stem\n    if name.isdigit():\n        return (0, int(name))\n    parts = re.split(r"(\\d+)", name)\n    return (1, tuple(int(p) if p.isdigit() else p.lower() for p in parts))\n\n\ndef init_runtime():\n    world_size = int(os.environ.get("WORLD_SIZE", "1"))\n    rank = int(os.environ.get("RANK", "0"))\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU is required.")\n\n    torch.cuda.set_device(local_rank)\n    if world_size > 1 and not dist.is_initialized():\n        dist.init_process_group(backend="nccl")\n\n    return rank, local_rank, world_size, torch.device(f"cuda:{local_rank}")\n\n\ndef destroy_distributed():\n    if dist.is_available() and dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef barrier():\n    if dist.is_available() and dist.is_initialized():\n        dist.barrier()\n\n\ndef broadcast_object(obj, rank: int):\n    if not (dist.is_available() and dist.is_initialized()):\n        return obj\n    payload = [obj if rank == 0 else None]\n    dist.broadcast_object_list(payload, src=0)\n    return payload[0]\n\n\ndef read_kaggle_secret(secret_name: str) -> str:\n    if not secret_name:\n        return ""\n    try:\n        from kaggle_secrets import UserSecretsClient\n        return UserSecretsClient().get_secret(secret_name) or ""\n    except Exception:\n        return ""\n\n\ndef make_storage_client(cfg: dict[str, Any]):\n    from google.cloud import storage\n\n    cred_file = str(cfg.get("gcs_credentials_file", "") or "").strip()\n    secret_name = str(cfg.get("gcs_credentials_json_secret_name", "") or "").strip()\n    cred_json = os.environ.get("GCS_CREDENTIALS_JSON", "").strip()\n\n    if not cred_json and secret_name:\n        cred_json = read_kaggle_secret(secret_name).strip()\n\n    if cred_json:\n        from google.oauth2 import service_account\n        info = json.loads(cred_json)\n        credentials = service_account.Credentials.from_service_account_info(info)\n        return storage.Client(project=credentials.project_id, credentials=credentials)\n\n    if cred_file:\n        return storage.Client.from_service_account_json(cred_file)\n\n    return storage.Client()\n\n\ndef batch_output_prefix(cfg: dict[str, Any], batch_id: str) -> str:\n    return (\n        f"{cfg[\'output_prefix\'].strip(\'/\')}/"\n        f"dataset={cfg[\'dataset_id\']}/"\n        f"batch={batch_id}/"\n        f"frame_profile={cfg[\'frame_profile\']}/"\n        f"extractor=vector-embedding/"\n        f"extractor_version={cfg[\'extractor_version\']}"\n    )\n\n\ndef discover_keyframes_dir(input_root: Path, batch_id: str) -> Path:\n    candidates: list[Path] = []\n    candidates.extend(input_root.glob(f"*Keyframes_{batch_id}/keyframes"))\n    candidates.extend(input_root.glob(f"*/*Keyframes_{batch_id}/keyframes"))\n    candidates.extend(input_root.glob(f"*/*/*Keyframes_{batch_id}/keyframes"))\n\n    unique = []\n    seen = set()\n    for p in candidates:\n        try:\n            resolved = p.resolve()\n        except Exception:\n            resolved = p\n        if str(resolved) not in seen and p.is_dir():\n            seen.add(str(resolved))\n            unique.append(p)\n\n    valid = []\n    for p in unique:\n        if any(child.is_dir() and child.name.startswith(batch_id + "_V") for child in p.iterdir()):\n            valid.append(p)\n\n    if not valid:\n        raise FileNotFoundError(\n            f"Cannot find local keyframes for {batch_id} under {input_root}."\n        )\n\n    valid.sort(key=lambda p: (len(p.parts), str(p)))\n    return valid[0]\n\n\ndef discover_jobs(cfg: dict[str, Any]) -> list[VideoJob]:\n    input_root = Path(cfg["input_root"])\n    jobs: list[VideoJob] = []\n    max_videos = cfg.get("max_videos_per_batch")\n\n    for batch_id in cfg["batches"]:\n        keyframes_dir = discover_keyframes_dir(input_root, batch_id)\n        video_dirs = sorted(\n            [p for p in keyframes_dir.iterdir() if p.is_dir() and p.name.startswith(batch_id + "_V")],\n            key=lambda p: natural_key(p.name),\n        )\n        if max_videos is not None:\n            video_dirs = video_dirs[: int(max_videos)]\n\n        for video_dir in video_dirs:\n            image_paths = sorted(\n                [p for p in video_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS],\n                key=natural_key,\n            )\n            if image_paths:\n                jobs.append(\n                    VideoJob(\n                        batch_id=batch_id,\n                        video_id=video_dir.name,\n                        video_dir=str(video_dir),\n                        image_paths=tuple(str(p) for p in image_paths),\n                    )\n                )\n\n    jobs.sort(key=lambda j: (j.batch_id, natural_key(j.video_id)))\n    return jobs\n\n\ndef get_existing_complete_videos(cfg: dict[str, Any], jobs: list[VideoJob]) -> dict[str, set[str]]:\n    if not cfg.get("upload_to_gcs", True) or not cfg.get("skip_existing", True):\n        return {batch: set() for batch in cfg["batches"]}\n\n    client = make_storage_client(cfg)\n    bucket = client.bucket(cfg["gcs_bucket"])\n    result: dict[str, set[str]] = {}\n\n    for batch_id in cfg["batches"]:\n        prefix = batch_output_prefix(cfg, batch_id) + "/"\n        emb = {\n            Path(blob.name).stem\n            for blob in bucket.list_blobs(prefix=prefix + "embeddings/")\n            if blob.name.endswith(".npy")\n        }\n        maps = {\n            Path(blob.name).stem\n            for blob in bucket.list_blobs(prefix=prefix + "map-keyframes/")\n            if blob.name.endswith(".csv")\n        }\n        discovered_ids = {j.video_id for j in jobs if j.batch_id == batch_id}\n        result[batch_id] = (emb & maps) & discovered_ids\n\n    return result\n\n\ndef greedy_partition(jobs: list[VideoJob], world_size: int) -> list[list[VideoJob]]:\n    partitions: list[list[VideoJob]] = [[] for _ in range(world_size)]\n    loads = [0] * world_size\n    for job in sorted(jobs, key=lambda j: (-j.num_frames, j.batch_id, j.video_id)):\n        idx = min(range(world_size), key=lambda r: (loads[r], r))\n        partitions[idx].append(job)\n        loads[idx] += job.num_frames\n    for part in partitions:\n        part.sort(key=lambda j: (j.batch_id, natural_key(j.video_id)))\n    return partitions\n\n\ndef frame_metadata(job: VideoJob, image_path: str, row_number: int) -> dict[str, Any]:\n    p = Path(image_path)\n    stem = p.stem\n    keyframe_number = int(stem) if stem.isdigit() else row_number\n    return {\n        "n": row_number,\n        "keyframe_number": keyframe_number,\n        "keyframe_id": f"{job.video_id}_{stem}",\n        "frame_filename": p.name,\n        "image_rel_path": f"{job.video_id}/{p.name}",\n    }\n\n\nclass LocalFrameDataset(Dataset):\n    def __init__(self, records: list[dict[str, Any]]):\n        self.records = records\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, idx):\n        path = self.records[idx]["image_path"]\n        try:\n            with Image.open(path) as image:\n                image = image.convert("RGB").copy()\n        except Exception as exc:\n            raise RuntimeError(f"Failed to read image: {path}") from exc\n        return image, idx\n\n\nclass SiglipCollator:\n    def __init__(self, image_processor):\n        self.image_processor = image_processor\n\n    def __call__(self, batch):\n        images, indices = zip(*batch)\n        inputs = self.image_processor(images=list(images), return_tensors="pt")\n        return dict(inputs), torch.tensor(indices, dtype=torch.int64)\n\n\ndef extract_siglip_embedding(outputs):\n    if torch.is_tensor(outputs):\n        return outputs\n    if getattr(outputs, "pooler_output", None) is not None:\n        return outputs.pooler_output\n    if getattr(outputs, "image_embeds", None) is not None:\n        return outputs.image_embeds\n    raise TypeError(f"Unsupported SigLIP image feature output: {type(outputs)}")\n\n\n# -------------------------\n# Minimal image-only Qwen3-VL embedder.\n# Mirrors the official Qwen3-VL-Embedding preprocessing + last-token pooling,\n# while allowing an 8B model to be sharded over both Kaggle GPUs.\n# -------------------------\nclass QwenImageEmbedder:\n    def __init__(\n        self,\n        model_name_or_path: str,\n        min_pixels: int,\n        max_pixels: int,\n        instruction: str,\n        device: torch.device,\n        sharded: bool,\n    ):\n        from dataclasses import dataclass\n        from transformers.modeling_outputs import ModelOutput\n        from transformers.models.qwen3_vl.modeling_qwen3_vl import (\n            Qwen3VLConfig,\n            Qwen3VLModel,\n            Qwen3VLPreTrainedModel,\n        )\n        from transformers.models.qwen3_vl.processing_qwen3_vl import Qwen3VLProcessor\n\n        @dataclass\n        class Qwen3VLForEmbeddingOutput(ModelOutput):\n            last_hidden_state: Optional[torch.FloatTensor] = None\n            attention_mask: Optional[torch.Tensor] = None\n\n        class Qwen3VLForEmbedding(Qwen3VLPreTrainedModel):\n            _checkpoint_conversion_mapping = {}\n            accepts_loss_kwargs = False\n            config: Qwen3VLConfig\n\n            def __init__(self, config):\n                super().__init__(config)\n                self.model = Qwen3VLModel(config)\n                self.post_init()\n\n            def get_input_embeddings(self):\n                return self.model.get_input_embeddings()\n\n            def set_input_embeddings(self, value):\n                self.model.set_input_embeddings(value)\n\n            def forward(\n                self,\n                input_ids=None,\n                attention_mask=None,\n                position_ids=None,\n                past_key_values=None,\n                inputs_embeds=None,\n                pixel_values=None,\n                pixel_values_videos=None,\n                image_grid_thw=None,\n                video_grid_thw=None,\n                cache_position=None,\n                **kwargs,\n            ):\n                outputs = self.model(\n                    input_ids=input_ids,\n                    pixel_values=pixel_values,\n                    pixel_values_videos=pixel_values_videos,\n                    image_grid_thw=image_grid_thw,\n                    video_grid_thw=video_grid_thw,\n                    position_ids=position_ids,\n                    attention_mask=attention_mask,\n                    past_key_values=past_key_values,\n                    inputs_embeds=inputs_embeds,\n                    cache_position=cache_position,\n                    **kwargs,\n                )\n                return Qwen3VLForEmbeddingOutput(\n                    last_hidden_state=outputs.last_hidden_state,\n                    attention_mask=attention_mask,\n                )\n\n        self.min_pixels = int(min_pixels)\n        self.max_pixels = int(max_pixels)\n        self.instruction = instruction\n        self.sharded = bool(sharded)\n\n        load_kwargs = dict(\n            torch_dtype=torch.float16,\n            low_cpu_mem_usage=True,\n            attn_implementation="sdpa",\n        )\n\n        if self.sharded:\n            if torch.cuda.device_count() < 2:\n                raise RuntimeError("Qwen3-VL-Embedding-8B sharded mode requires 2 visible GPUs.")\n            # Reserve activation headroom on each 16GB T4.\n            max_memory = {}\n            for i in range(torch.cuda.device_count()):\n                total_gib = torch.cuda.get_device_properties(i).total_memory / (1024**3)\n                reserve = 2.5\n                usable = max(8, int(total_gib - reserve))\n                max_memory[i] = f"{usable}GiB"\n            load_kwargs["device_map"] = "balanced_low_0"\n            load_kwargs["max_memory"] = max_memory\n\n        self.model = Qwen3VLForEmbedding.from_pretrained(\n            model_name_or_path,\n            trust_remote_code=True,\n            **load_kwargs,\n        )\n\n        if not self.sharded:\n            self.model = self.model.to(device)\n\n        self.model.eval()\n        self.processor = Qwen3VLProcessor.from_pretrained(\n            model_name_or_path,\n            padding_side="right",\n        )\n\n        if self.sharded:\n            print("Qwen hf_device_map:", getattr(self.model, "hf_device_map", None))\n\n    @staticmethod\n    def _pooling_last(hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        flipped = attention_mask.flip(dims=[1])\n        last_one_positions = flipped.argmax(dim=1)\n        col = attention_mask.shape[1] - last_one_positions - 1\n        row = torch.arange(hidden_state.shape[0], device=hidden_state.device)\n        return hidden_state[row, col]\n\n    def _input_device(self) -> torch.device:\n        try:\n            return self.model.device\n        except Exception:\n            return next(self.model.parameters()).device\n\n    @torch.inference_mode()\n    def encode_paths(self, paths: list[str]) -> torch.Tensor:\n        from qwen_vl_utils.vision_process import process_vision_info\n\n        conversations = []\n        for path in paths:\n            conversations.append(\n                [\n                    {\n                        "role": "system",\n                        "content": [{"type": "text", "text": self.instruction}],\n                    },\n                    {\n                        "role": "user",\n                        "content": [\n                            {\n                                "type": "image",\n                                "image": "file://" + path,\n                                "min_pixels": self.min_pixels,\n                                "max_pixels": self.max_pixels,\n                            }\n                        ],\n                    },\n                ]\n            )\n\n        text = self.processor.apply_chat_template(\n            conversations,\n            add_generation_prompt=True,\n            tokenize=False,\n        )\n\n        images, video_inputs, video_kwargs = process_vision_info(\n            conversations,\n            image_patch_size=16,\n            return_video_metadata=True,\n            return_video_kwargs=True,\n        )\n\n        if video_inputs is not None:\n            videos, video_metadata = zip(*video_inputs)\n            videos, video_metadata = list(videos), list(video_metadata)\n        else:\n            videos, video_metadata = None, None\n\n        inputs = self.processor(\n            text=text,\n            images=images,\n            videos=videos,\n            video_metadata=video_metadata,\n            truncation=True,\n            max_length=8192,\n            padding=True,\n            do_resize=False,\n            return_tensors="pt",\n            **video_kwargs,\n        )\n\n        input_device = self._input_device()\n        inputs = {\n            k: (v.to(input_device) if hasattr(v, "to") else v)\n            for k, v in inputs.items()\n        }\n\n        outputs = self.model(**inputs)\n        emb = self._pooling_last(outputs.last_hidden_state, inputs["attention_mask"])\n        return F.normalize(emb.float(), p=2, dim=-1)\n\n\ndef atomic_save_npy(path: Path, array: np.ndarray):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with tmp.open("wb") as f:\n        np.save(f, array)\n    os.replace(tmp, path)\n\n\ndef save_video_outputs(cfg, local_root, job, embedding_chunks):\n    embeddings = np.concatenate(embedding_chunks, axis=0)\n    if embeddings.shape[0] != job.num_frames:\n        raise RuntimeError(\n            f"{job.video_id}: embedding rows={embeddings.shape[0]} but frames={job.num_frames}"\n        )\n\n    save_dtype = np.float16 if cfg["save_dtype"] == "float16" else np.float32\n    embeddings = embeddings.astype(save_dtype, copy=False)\n\n    batch_dir = local_root / job.batch_id\n    emb_path = batch_dir / "embeddings" / f"{job.video_id}.npy"\n    map_path = batch_dir / "map-keyframes" / f"{job.video_id}.csv"\n    atomic_save_npy(emb_path, embeddings)\n\n    rows = [\n        frame_metadata(job, image_path, i + 1)\n        for i, image_path in enumerate(job.image_paths)\n    ]\n    map_path.parent.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(rows).to_csv(map_path, index=False)\n    return emb_path, map_path, embeddings.shape\n\n\ndef build_records(jobs):\n    records = []\n    job_by_video = {}\n    for job in jobs:\n        if job.video_id in job_by_video:\n            raise ValueError(f"Duplicate video_id: {job.video_id}")\n        job_by_video[job.video_id] = job\n        for image_path in job.image_paths:\n            records.append(\n                {\n                    "batch_id": job.batch_id,\n                    "video_id": job.video_id,\n                    "image_path": image_path,\n                }\n            )\n    return records, job_by_video\n\n\ndef upload_one(bucket, local_path: Path, object_key: str):\n    blob = bucket.blob(object_key)\n    blob.upload_from_filename(str(local_path))\n\n\ndef upload_rank_outputs(cfg, paths):\n    if not cfg.get("upload_to_gcs", True) or not paths:\n        return\n    client = make_storage_client(cfg)\n    bucket = client.bucket(cfg["gcs_bucket"])\n    workers = max(1, int(cfg.get("upload_workers_per_rank", 4)))\n\n    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="gcs-upload") as pool:\n        futures = {\n            pool.submit(upload_one, bucket, local_path, object_key): (local_path, object_key)\n            for object_key, local_path in paths\n        }\n        for future in tqdm(\n            as_completed(futures),\n            total=len(futures),\n            desc="Uploading outputs",\n            leave=False,\n        ):\n            future.result()\n\n\ndef write_json(path: Path, obj: dict):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef peak_memory_by_gpu():\n    out = {}\n    for i in range(torch.cuda.device_count()):\n        out[f"cuda:{i}"] = round(torch.cuda.max_memory_allocated(i) / 1024**2, 2)\n    return out\n\n\ndef finalize_batch_metadata(cfg, local_root, jobs_all, existing, elapsed):\n    client = make_storage_client(cfg) if cfg.get("upload_to_gcs", True) else None\n    bucket = client.bucket(cfg["gcs_bucket"]) if client else None\n\n    for batch_id in cfg["batches"]:\n        batch_jobs = [j for j in jobs_all if j.batch_id == batch_id]\n        batch_dir = local_root / batch_id\n        batch_dir.mkdir(parents=True, exist_ok=True)\n\n        produced_embs = sorted((batch_dir / "embeddings").glob("*.npy")) if (batch_dir / "embeddings").exists() else []\n        produced_maps = sorted((batch_dir / "map-keyframes").glob("*.csv")) if (batch_dir / "map-keyframes").exists() else []\n\n        model_info = {\n            "model_key": cfg["model_key"],\n            "checkpoint": cfg["checkpoint"],\n            "backend": cfg["backend"],\n            "embedding_dimension": int(cfg["embedding_dim"]),\n            "normalized": True,\n            "normalization": "L2",\n            "inference_dtype": "float16",\n            "saved_dtype": cfg["save_dtype"],\n            "qwen_instruction": cfg.get("qwen_instruction"),\n            "qwen_min_pixels": cfg.get("qwen_min_pixels"),\n            "qwen_max_pixels": cfg.get("qwen_max_pixels"),\n            "launch_mode": cfg["launch_mode"],\n            "extractor": "vector-embedding",\n            "extractor_version": cfg["extractor_version"],\n        }\n\n        summary = {\n            "status": "PARTIAL_SUCCESS" if cfg.get("max_videos_per_batch") is not None else "SUCCESS",\n            "batch_id": batch_id,\n            "videos_discovered": len(batch_jobs),\n            "frames_discovered": int(sum(j.num_frames for j in batch_jobs)),\n            "videos_skipped_existing_gcs": len(existing.get(batch_id, set())),\n            "videos_produced_this_session": len(produced_embs),\n            "maps_produced_this_session": len(produced_maps),\n            "elapsed_seconds_global": round(elapsed, 3),\n            "gcs_prefix": f"gs://{cfg[\'gcs_bucket\']}/{batch_output_prefix(cfg, batch_id)}/",\n        }\n\n        model_info_path = batch_dir / "model_info.json"\n        summary_path = batch_dir / "summary.json"\n        write_json(model_info_path, model_info)\n        write_json(summary_path, summary)\n\n        if bucket is not None:\n            prefix = batch_output_prefix(cfg, batch_id) + "/"\n            upload_one(bucket, model_info_path, prefix + "model_info.json")\n            upload_one(bucket, summary_path, prefix + "summary.json")\n            marker = "_PARTIAL_SUCCESS" if cfg.get("max_videos_per_batch") is not None else "_SUCCESS"\n            bucket.blob(prefix + marker).upload_from_string("", content_type="text/plain")\n\n\ndef append_embedding_groups(\n    emb_np,\n    indices,\n    records,\n    job_by_video,\n    state,\n    cfg,\n    local_root,\n    created_uploads,\n):\n    current_video = state["current_video"]\n    current_chunks = state["current_chunks"]\n\n    batch_video_ids = [records[i]["video_id"] for i in indices]\n    start = 0\n\n    def flush():\n        nonlocal current_video, current_chunks\n        if current_video is None:\n            return\n        job = job_by_video[current_video]\n        emb_path, map_path, shape = save_video_outputs(\n            cfg, local_root, job, current_chunks\n        )\n        prefix = batch_output_prefix(cfg, job.batch_id) + "/"\n        created_uploads.append((prefix + f"embeddings/{job.video_id}.npy", emb_path))\n        created_uploads.append((prefix + f"map-keyframes/{job.video_id}.csv", map_path))\n        state["processed_videos"] += 1\n        state["processed_frames"] += int(shape[0])\n        current_video = None\n        current_chunks = []\n\n    while start < len(indices):\n        vid = batch_video_ids[start]\n        end = start + 1\n        while end < len(indices) and batch_video_ids[end] == vid:\n            end += 1\n\n        if current_video is None:\n            current_video = vid\n        elif vid != current_video:\n            flush()\n            current_video = vid\n\n        current_chunks.append(emb_np[start:end])\n        start = end\n\n    state["current_video"] = current_video\n    state["current_chunks"] = current_chunks\n    return flush\n\n\ndef run_siglip(cfg, records, job_by_video, device, rank, workers, local_root):\n    from transformers import AutoImageProcessor, AutoModel\n\n    image_processor = AutoImageProcessor.from_pretrained(cfg["checkpoint"], use_fast=True)\n    model = AutoModel.from_pretrained(\n        cfg["checkpoint"],\n        torch_dtype=torch.float16,\n    ).to(device).eval()\n\n    dataset = LocalFrameDataset(records)\n    loader_kwargs = dict(\n        dataset=dataset,\n        batch_size=int(cfg["batch_size_per_gpu"]),\n        shuffle=False,\n        num_workers=workers,\n        pin_memory=True,\n        persistent_workers=(workers > 0),\n        collate_fn=SiglipCollator(image_processor),\n        drop_last=False,\n    )\n    if workers > 0:\n        loader_kwargs["prefetch_factor"] = int(cfg.get("prefetch_factor", 2))\n    loader = DataLoader(**loader_kwargs)\n\n    state = {\n        "current_video": None,\n        "current_chunks": [],\n        "processed_videos": 0,\n        "processed_frames": 0,\n    }\n    created_uploads = []\n\n    pbar = tqdm(loader, total=len(loader), desc=f"{cfg[\'model_key\']} rank{rank}", position=rank)\n    last_flush = None\n    with torch.inference_mode():\n        for inputs, sample_indices in pbar:\n            gpu_inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}\n            with torch.autocast("cuda", dtype=torch.float16):\n                outputs = model.get_image_features(**gpu_inputs)\n                emb = extract_siglip_embedding(outputs)\n            emb_np = F.normalize(emb.float(), p=2, dim=-1).cpu().numpy()\n            indices = sample_indices.tolist()\n            last_flush = append_embedding_groups(\n                emb_np, indices, records, job_by_video, state, cfg, local_root, created_uploads\n            )\n\n    if last_flush:\n        last_flush()\n    return state, created_uploads\n\n\ndef qwen_encode_adaptive(embedder, paths: list[str]) -> torch.Tensor:\n    """Retry a Qwen batch at half size if activations OOM."""\n    try:\n        return embedder.encode_paths(paths)\n    except torch.OutOfMemoryError:\n        if len(paths) <= 1:\n            raise\n        torch.cuda.empty_cache()\n        mid = len(paths) // 2\n        print(f"[OOM fallback] splitting Qwen batch {len(paths)} -> {mid} + {len(paths)-mid}")\n        left = qwen_encode_adaptive(embedder, paths[:mid])\n        right = qwen_encode_adaptive(embedder, paths[mid:])\n        return torch.cat([left, right], dim=0)\n\n\ndef run_qwen(cfg, records, job_by_video, device, rank, local_root):\n    embedder = QwenImageEmbedder(\n        model_name_or_path=cfg["checkpoint"],\n        min_pixels=int(cfg["qwen_min_pixels"]),\n        max_pixels=int(cfg["qwen_max_pixels"]),\n        instruction=cfg["qwen_instruction"],\n        device=device,\n        sharded=(cfg["launch_mode"] == "sharded"),\n    )\n\n    batch_size = int(cfg["batch_size_per_gpu"])\n    state = {\n        "current_video": None,\n        "current_chunks": [],\n        "processed_videos": 0,\n        "processed_frames": 0,\n    }\n    created_uploads = []\n    last_flush = None\n\n    starts = range(0, len(records), batch_size)\n    for start in tqdm(\n        starts,\n        total=(len(records) + batch_size - 1) // batch_size,\n        desc=f"{cfg[\'model_key\']} rank{rank}",\n        position=rank,\n    ):\n        end = min(len(records), start + batch_size)\n        paths = [records[i]["image_path"] for i in range(start, end)]\n        emb = qwen_encode_adaptive(embedder, paths)\n        emb_np = emb.cpu().numpy()\n        indices = list(range(start, end))\n        last_flush = append_embedding_groups(\n            emb_np, indices, records, job_by_video, state, cfg, local_root, created_uploads\n        )\n\n    if last_flush:\n        last_flush()\n    return state, created_uploads\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n    cfg = json.loads(Path(args.config).read_text(encoding="utf-8"))\n\n    rank, local_rank, world_size, device = init_runtime()\n\n    if cfg["launch_mode"] == "sharded" and world_size != 1:\n        raise RuntimeError("Sharded mode must be launched with plain python, not torchrun.")\n\n    cpu_count = os.cpu_count() or 4\n    configured_workers = cfg.get("num_workers_per_gpu")\n    if configured_workers is None:\n        workers = max(1, min(4, cpu_count // max(1, world_size)))\n    else:\n        workers = max(0, int(configured_workers))\n\n    torch.set_num_threads(1)\n    torch.backends.cudnn.benchmark = True\n    torch.backends.cuda.matmul.allow_tf32 = True\n\n    for i in range(torch.cuda.device_count()):\n        torch.cuda.reset_peak_memory_stats(i)\n\n    if rank == 0:\n        print("Model:", cfg["model_key"], "| checkpoint:", cfg["checkpoint"])\n        print("Launch mode:", cfg["launch_mode"], "| world_size:", world_size)\n        for i in range(torch.cuda.device_count()):\n            print(f"  cuda:{i}: {torch.cuda.get_device_name(i)}")\n\n    all_jobs = discover_jobs(cfg)\n    existing = None\n    if rank == 0:\n        existing = get_existing_complete_videos(cfg, all_jobs)\n        print("Existing complete videos:", {k: len(v) for k, v in existing.items()})\n    existing = broadcast_object(existing, rank)\n\n    remaining_jobs = [\n        j for j in all_jobs\n        if j.video_id not in existing.get(j.batch_id, set())\n    ]\n\n    partitions = greedy_partition(remaining_jobs, world_size)\n    my_jobs = partitions[rank]\n\n    if rank == 0:\n        print("Planned frames per process:", [sum(j.num_frames for j in p) for p in partitions])\n\n    local_root = Path(cfg["local_output_root"])\n    local_root.mkdir(parents=True, exist_ok=True)\n\n    if not my_jobs:\n        barrier()\n        if rank == 0:\n            finalize_batch_metadata(cfg, local_root, all_jobs, existing, elapsed=0.0)\n        barrier()\n        destroy_distributed()\n        return\n\n    records, job_by_video = build_records(my_jobs)\n    started = time.perf_counter()\n\n    if cfg["backend"] == "siglip2":\n        state, created_uploads = run_siglip(\n            cfg, records, job_by_video, device, rank, workers, local_root\n        )\n    elif cfg["backend"] == "qwen3_vl":\n        state, created_uploads = run_qwen(\n            cfg, records, job_by_video, device, rank, local_root\n        )\n    else:\n        raise ValueError(f"Unknown backend: {cfg[\'backend\']}")\n\n    if cfg["backend"] == "siglip2":\n        torch.cuda.synchronize(device)\n    else:\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n    upload_started = time.perf_counter()\n    upload_rank_outputs(cfg, created_uploads)\n    upload_seconds = time.perf_counter() - upload_started\n    elapsed = time.perf_counter() - started\n\n    rank_metrics = {\n        "rank": rank,\n        "local_rank": local_rank,\n        "model_key": cfg["model_key"],\n        "videos_processed": state["processed_videos"],\n        "frames_processed": state["processed_frames"],\n        "elapsed_seconds": round(elapsed, 3),\n        "upload_seconds": round(upload_seconds, 3),\n        "frames_per_second_including_upload": round(state["processed_frames"] / elapsed, 3) if elapsed else 0,\n        "peak_allocated_mb_by_gpu": peak_memory_by_gpu(),\n        "batch_size": int(cfg["batch_size_per_gpu"]),\n    }\n    write_json(local_root / f"rank_{rank}_metrics.json", rank_metrics)\n    print(json.dumps(rank_metrics, indent=2))\n\n    barrier()\n\n    if rank == 0:\n        metric_files = sorted(local_root.glob("rank_*_metrics.json"))\n        metrics = [json.loads(p.read_text(encoding="utf-8")) for p in metric_files]\n        global_elapsed = max((float(m["elapsed_seconds"]) for m in metrics), default=elapsed)\n        write_json(local_root / "all_rank_metrics.json", {"ranks": metrics})\n        finalize_batch_metadata(cfg, local_root, all_jobs, existing, global_elapsed)\n\n        print("\\nFinal GCS prefixes:")\n        for batch_id in cfg["batches"]:\n            print(f"gs://{cfg[\'gcs_bucket\']}/{batch_output_prefix(cfg, batch_id)}/")\n\n    barrier()\n    destroy_distributed()\n\n\nif __name__ == "__main__":\n    main()\n'

Path(SCRIPT_PATH).write_text(script_text, encoding="utf-8")
print("Worker written:", SCRIPT_PATH)
print("Worker chars:", len(script_text))


## 5. Build one runtime config per model

Mỗi model có local output riêng và GCS `extractor_version` riêng. Vì vậy `SKIP_EXISTING=True` sẽ resume độc lập cho từng model.


In [ ]:
CONFIG_PATHS = {}

for model_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[model_key]
    config_path = f"/kaggle/working/{model_key}_embedding_config.json"

    cfg = {
        "model_key": model_key,
        "backend": spec["backend"],
        "checkpoint": spec["checkpoint"],
        "embedding_dim": spec["embedding_dim"],
        "batch_size_per_gpu": spec["batch_size_per_gpu"],
        "launch_mode": spec["launch_mode"],
        "extractor_version": spec["extractor_version"],

        "input_root": INPUT_ROOT,
        "batches": BATCHES,
        "max_videos_per_batch": MAX_VIDEOS_PER_BATCH,

        "num_workers_per_gpu": NUM_WORKERS_PER_GPU,
        "prefetch_factor": PREFETCH_FACTOR,
        "save_dtype": SAVE_DTYPE,

        "qwen_min_pixels": QWEN_MIN_PIXELS,
        "qwen_max_pixels": QWEN_MAX_PIXELS,
        "qwen_instruction": QWEN_DOCUMENT_INSTRUCTION,

        "gcs_bucket": GCS_BUCKET,
        "dataset_id": DATASET_ID,
        "frame_profile": FRAME_PROFILE,
        "output_prefix": OUTPUT_PREFIX,

        "upload_to_gcs": UPLOAD_TO_GCS,
        "skip_existing": SKIP_EXISTING,
        "upload_workers_per_rank": UPLOAD_WORKERS_PER_RANK,
        "gcs_credentials_json_secret_name": GCS_CREDENTIALS_JSON_SECRET_NAME,
        "gcs_credentials_file": GCS_CREDENTIALS_FILE,

        "local_output_root": f"{LOCAL_OUTPUT_ROOT_BASE}/{model_key}",
    }

    Path(config_path).write_text(
        json.dumps(cfg, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    CONFIG_PATHS[model_key] = config_path

    print(f"\n[{model_key}]")
    print(Path(config_path).read_text())


## 6. Smoke test strongly recommended

Trước full run:
1. đặt `MAX_VIDEOS_PER_BATCH = 1`,
2. chạy lại cell Configuration + Build configs,
3. chạy cell bên dưới cho cả 3 model,
4. kiểm tra shape và GCS prefixes.

Sau đó đổi lại `MAX_VIDEOS_PER_BATCH = None`. Với `SKIP_EXISTING=True`, video đã có đủ `.npy` + `.csv` trong **đúng extractor_version của model đó** sẽ được skip.

Qwen 8B tải model lớn hơn đáng kể; chạy tuần tự giúp giải phóng VRAM hoàn toàn giữa các model.


## 7. Run all selected models sequentially on Kaggle GPU(s)

- `siglip2`: `torchrun` với 1 process/GPU.
- `qwen3-vl-2b`: `torchrun` với 1 model replica/GPU.
- `qwen3-vl-8b`: plain Python, 1 process, model sharded trên cả 2 GPU.


In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN

for model_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[model_key]
    config_path = CONFIG_PATHS[model_key]

    print("\n" + "=" * 90)
    print("RUNNING:", model_key, "|", spec["checkpoint"])
    print("=" * 90)

    if spec["launch_mode"] == "replicated":
        nproc = min(2, GPU_COUNT)
        cmd = [
            "torchrun",
            "--standalone",
            f"--nproc_per_node={nproc}",
            SCRIPT_PATH,
            "--config",
            config_path,
        ]
    elif spec["launch_mode"] == "sharded":
        cmd = [
            sys.executable,
            SCRIPT_PATH,
            "--config",
            config_path,
        ]
    else:
        raise ValueError(spec["launch_mode"])

    print("Command:", " ".join(cmd))
    subprocess.run(cmd, env=env, check=True)

    # The subprocess exits here, so model VRAM is released before the next model.
    print("DONE:", model_key)


## 8. Validate local outputs produced in this Kaggle session

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

for model_key in MODELS_TO_RUN:
    expected_dim = MODEL_SPECS[model_key]["embedding_dim"]
    root = Path(LOCAL_OUTPUT_ROOT_BASE) / model_key

    total_videos = 0
    total_vectors = 0

    print("\n" + "=" * 80)
    print(model_key, "| expected dim =", expected_dim)

    for batch in BATCHES:
        emb_dir = root / batch / "embeddings"
        map_dir = root / batch / "map-keyframes"

        npy_files = sorted(emb_dir.glob("*.npy")) if emb_dir.exists() else []
        print(f"{batch}: produced this session = {len(npy_files)} videos")

        for npy_path in npy_files:
            csv_path = map_dir / f"{npy_path.stem}.csv"
            assert csv_path.exists(), f"Missing map CSV: {csv_path}"

            emb = np.load(npy_path, mmap_mode="r")
            df = pd.read_csv(csv_path)

            assert emb.ndim == 2
            assert emb.shape[0] == len(df), (npy_path.name, emb.shape, len(df))
            assert emb.shape[1] == expected_dim, (npy_path.name, emb.shape, expected_dim)

            total_videos += 1
            total_vectors += emb.shape[0]

    print("Validated videos:", total_videos)
    print("Validated vectors:", total_vectors)


## 9. Expected GCS layout

Quan trọng: Qwen được upload đúng hai version bạn yêu cầu:
- `extractor_version=qwen3-vl-2b/`
- `extractor_version=qwen3-vl-8b/`


In [ ]:
for batch in BATCHES:
    print(f"\n### {batch}")
    for model_key in MODELS_TO_RUN:
        version = MODEL_SPECS[model_key]["extractor_version"]
        prefix = (
            f"gs://{GCS_BUCKET}/{OUTPUT_PREFIX}/"
            f"dataset={DATASET_ID}/batch={batch}/"
            f"frame_profile={FRAME_PROFILE}/"
            f"extractor=vector-embedding/"
            f"extractor_version={version}/"
        )
        print(f"{model_key:14s} -> {prefix}")

print(
    "\nInside each model/batch prefix:\n"
    "  embeddings/Lxx_Vxxx.npy\n"
    "  map-keyframes/Lxx_Vxxx.csv\n"
    "  model_info.json\n"
    "  summary.json\n"
    "  _SUCCESS  # full run; smoke test uses _PARTIAL_SUCCESS"
)


## Kaggle T4×2 tuning and fair evaluation notes

- **SigLIP2** starts at batch 512/GPU, matching the original notebook.
- **Qwen 2B** starts at batch 8/GPU and uses both T4s independently. If a particular batch OOMs, it is automatically split.
- **Qwen 8B** is not weight-quantized: it uses FP16 and shards the model over both T4s. This is slower than two replicas, but preserves the original model better for a quality comparison.
- Qwen defaults to `QWEN_MAX_PIXELS=256*256` so the comparison does not silently give Qwen much higher input resolution than SigLIP2. For a second experiment, you can increase this and label it as a separate setting.
- Keep full embedding dimensions (768 / 2048 / 4096) for the first quality benchmark. Test MRL/truncated dimensions only as a separate latency/storage experiment.
- Use the same frame set, same query set, same relevance ground truth, same ANN settings (or exact cosine for a small benchmark), and compare **Recall@K / mAP / nDCG**. Raw cosine magnitudes are not comparable across model families.
